# Merge NCIt + PhenX and split into train / val / test

Combine NCIt training file with the PhenX REDCap pairs, then split **80 / 10 / 10** into train / validation / test.

- **train** — what the model learns from
- **val** — watched during training for best-checkpoint selection
- **test** — held out for *hyperparameter* decisions after training.

The split is **grouped by anchor**: all rows with the same `variable_para` go to the same split,
so a near-duplicate anchor can't leak between train and test.

**v2 change**: each row now carries a `source` column (`ncit` / `phenx`) so downstream subsets can
verify coverage directly. The split itself is unchanged — same seed, same grouping keys — so the
train/val/test row membership is identical to v1; only the extra column is new.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

NCIT_CSV  = "/home/fan1/notebooks/ncit_training.csv"                 # freshly generated (甲 format)
PHENX_CSV = "/home/fan1/notebooks/phenx_data/phenx_redcap_pairs.csv" # freshly generated (甲 format)

OUT_DIR = "/home/fan1/notebooks/single_stage_phenx_ncit_data_0818"  # new dir for the 0818 (甲) splits
os.makedirs(OUT_DIR, exist_ok=True)

COLS = ["variable_para", "alternate_variable_para"]
print("output ->", OUT_DIR)

output -> /home/fan1/notebooks/single_stage_phenx_ncit_data_0818


## Load both sources (two training columns + a `source` tag)

In [2]:
ncit  = pd.read_csv(NCIT_CSV,  usecols=COLS)
phenx = pd.read_csv(PHENX_CSV, usecols=COLS)

# tag provenance BEFORE merging, so it survives the split and lands in the CSVs
ncit["source"]  = "ncit"
phenx["source"] = "phenx"

print(f"NCIt rows  : {len(ncit):,}")
print(f"PhenX rows : {len(phenx):,}")
print(f"raw total  : {len(ncit) + len(phenx):,}")

NCIt rows  : 1,370,164
PhenX rows : 21,541
raw total  : 1,391,705


## Drop empty pairs

We check and drop any row where either side is empty.

In [3]:
def clean(df, name):
    before = len(df)
    df = df.copy()
    for c in COLS:
        df[c] = df[c].fillna("").astype(str).str.strip()
    df = df[(df["variable_para"] != "") & (df["alternate_variable_para"] != "")]
    print(f"{name}: {before:,} -> {len(df):,}  (dropped {before - len(df):,} empty)")
    return df.reset_index(drop=True)

ncit  = clean(ncit,  "NCIt ")
phenx = clean(phenx, "PhenX")

merged = pd.concat([ncit, phenx], ignore_index=True)
print(f"\nmerged (clean): {len(merged):,}")

# reference distribution of the FULL merged data — downstream subsets are compared against this
merged_pct = merged["source"].value_counts(normalize=True) * 100
print("\nmerged source distribution (reference for subset sanity checks):")
for s, p in merged_pct.items():
    print(f"  {s:<6}: {merged['source'].value_counts()[s]:>9,}  ({p:.3f}%)")

NCIt : 1,370,164 -> 1,370,164  (dropped 0 empty)
PhenX: 21,541 -> 21,541  (dropped 0 empty)

merged (clean): 1,391,705

merged source distribution (reference for subset sanity checks):
  ncit  : 1,370,164  (98.452%)
  phenx :    21,541  (1.548%)


## Split 80 / 10 / 10, grouped by anchor

`GroupShuffleSplit` keeps every row sharing an anchor together. We first peel off 20% (val+test),
then halve that into val and test.

Note: GSS randomizes **which anchor group goes to which split**, but the returned indices are
ascending — so within each saved CSV the physical row order is still NCIt block first, PhenX block
last. Any downstream `select(range(n))` MUST shuffle first (see the training notebook).

In [4]:
groups = merged["variable_para"]

# 80% train  /  20% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(gss1.split(merged, groups=groups))
train = merged.iloc[train_idx].reset_index(drop=True)
temp  = merged.iloc[temp_idx].reset_index(drop=True)

# temp -> 50/50 val/test  (i.e. 10% / 10% overall)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp, groups=temp["variable_para"]))
val  = temp.iloc[val_idx].reset_index(drop=True)
test = temp.iloc[test_idx].reset_index(drop=True)

n = len(merged)
print(f"train: {len(train):>9,}  ({len(train)/n*100:.1f}%)")
print(f"val  : {len(val):>9,}  ({len(val)/n*100:.1f}%)")
print(f"test : {len(test):>9,}  ({len(test)/n*100:.1f}%)")

# sanity: no anchor shared across splits
a_tr, a_va, a_te = set(train["variable_para"]), set(val["variable_para"]), set(test["variable_para"])
print("\nanchor overlap train/val :", len(a_tr & a_va))
print("anchor overlap train/test:", len(a_tr & a_te))
print("anchor overlap val/test  :", len(a_va & a_te))

# source distribution per split vs the merged reference
print("\nsource distribution per split (expect each close to the merged reference):")
for name, df in [("merged", merged), ("train", train), ("val", val), ("test", test)]:
    vc = df["source"].value_counts()
    pn, pp = vc.get("ncit", 0), vc.get("phenx", 0)
    print(f"  {name:<7}: ncit {pn:>9,} ({pn/len(df)*100:.3f}%) | phenx {pp:>7,} ({pp/len(df)*100:.3f}%)")

train: 1,114,017  (80.0%)
val  :   138,670  (10.0%)
test :   139,018  (10.0%)

anchor overlap train/val : 0
anchor overlap train/test: 0
anchor overlap val/test  : 0

source distribution per split (expect each close to the merged reference):
  merged : ncit 1,370,164 (98.452%) | phenx  21,541 (1.548%)
  train  : ncit 1,096,702 (98.446%) | phenx  17,315 (1.554%)
  val    : ncit   136,583 (98.495%) | phenx   2,087 (1.505%)
  test   : ncit   136,879 (98.461%) | phenx   2,139 (1.539%)


## Save

In [5]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print(f"saved {len(df):>9,} -> {path}   (columns: {list(df.columns)})")

# a quick peek so we can eyeball the format one more time
print("\nexample train row:")
print("  anchor  :", train["variable_para"].iloc[0][:150])
print("  positive:", train["alternate_variable_para"].iloc[0][:150])
print("  source  :", train["source"].iloc[0])

saved 1,114,017 -> /home/fan1/notebooks/single_stage_phenx_ncit_data_0818/train.csv   (columns: ['variable_para', 'alternate_variable_para', 'source'])


saved   138,670 -> /home/fan1/notebooks/single_stage_phenx_ncit_data_0818/val.csv   (columns: ['variable_para', 'alternate_variable_para', 'source'])


saved   139,018 -> /home/fan1/notebooks/single_stage_phenx_ncit_data_0818/test.csv   (columns: ['variable_para', 'alternate_variable_para', 'source'])

example train row:
  anchor  : name: PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ST ELEVATION MYOCARDIAL INFARCTION (STEMI) (STABLE, >12 HRS FROM SYMPTOM ONSET) | description: A pe
  positive: name: Percutaneous Coronary Intervention for ST Elevation Myocardial Infarction-Stable-Over 12 Hours From Symptom Onset | description: A percutaneous 
  source  : ncit
